# Week 3 — Pain Expression Detection with YOLOv8

**Task:** Object Detection — detect faces and classify pain level simultaneously  
**Dataset:** SZU-EmoDage (Dynamic_faces + Emotional_faces)  
**Model:** YOLOv8n (fine-tuned)  
**Classes:** No_Pain · Mild · Moderate · Severe  
**Metric:** mAP@0.5, mAP@0.5:0.95

---
**Pipeline:**
1. Extract dataset from Drive ZIP
2. Auto-annotate with MediaPipe face detector → YOLO bounding boxes
3. Build YOLO dataset (train / val / test split)
4. Train YOLOv8
5. Evaluate mAP
6. Visualize predictions
7. Save results to Drive

## Cell 1 — Install Dependencies

In [ ]:
!pip install ultralytics -q
print('Libraries installed.')

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

## Cell 3 — Configuration

> **Change `DRIVE_ZIP`** to the path of your dataset ZIP on Google Drive.

In [ ]:
from pathlib import Path

# ── CHANGE THIS to your Drive ZIP path ──────────────────────────────────────
DRIVE_ZIP = '/content/drive/MyDrive/pain_dataset_new.zip'
# ────────────────────────────────────────────────────────────────────────────

WORK_DIR    = Path('/content/pain_detection')
YOLO_DIR    = Path('/content/yolo_dataset')
RESULTS_DIR = Path('/content/pain_yolo_results')

CLASSES = ['No_Pain', 'Mild', 'Moderate', 'Severe']
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}

EMOTION_TO_PAIN = {
    'neutral':   'No_Pain',
    'happiness': 'No_Pain',
    'hapiness':  'No_Pain',
    'sadness':   'Mild',
    'fear':      'Moderate',
    'surprise':  'Moderate',
    'suprise':   'Moderate',
    'surpris':   'Moderate',
    'anger':     'Severe',
    'disgust':   'Severe',
    'disgest':   'Severe',
}

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
# remaining 0.15 -> test

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

print('Configuration ready.')
print(f'Classes: {CLASSES}')

## Cell 4 — Extract Dataset

In [ ]:
import zipfile, shutil

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)

print(f'Extracting {DRIVE_ZIP} ...')
with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
    z.extractall(WORK_DIR)
print('Extraction complete.')

FRAMES_DIR = WORK_DIR / 'images' / 'extracted_frames'
EMO_DIR    = WORK_DIR / 'images' / 'Emotional_faces' / 'Emotional_faces'

print(f'extracted_frames exists : {FRAMES_DIR.exists()}')
print(f'Emotional_faces exists  : {EMO_DIR.exists()}')

## Cell 5 — Collect All Image Paths with Pain Labels

In [ ]:
from collections import Counter

all_samples = []   # list of (image_path, pain_label)

# Source 1: extracted_frames/<PainLevel>/*.jpg
if FRAMES_DIR.exists():
    for label in CLASSES:
        label_dir = FRAMES_DIR / label
        if label_dir.exists():
            for p in label_dir.iterdir():
                if p.suffix.lower() in IMG_EXTS:
                    all_samples.append((p, label))
    print(f'Loaded from extracted_frames')
else:
    print('WARNING: extracted_frames not found')

# Source 2: Emotional_faces/<subject>/<emotion>.jpg
if EMO_DIR.exists():
    for subj_dir in EMO_DIR.iterdir():
        if not subj_dir.is_dir():
            continue
        for img_path in subj_dir.iterdir():
            if img_path.suffix.lower() not in IMG_EXTS:
                continue
            pain = EMOTION_TO_PAIN.get(img_path.stem.lower())
            if pain:
                all_samples.append((img_path, pain))
    print(f'Loaded from Emotional_faces')
else:
    print('WARNING: Emotional_faces not found')

counts = Counter(label for _, label in all_samples)
print(f'\nTotal images collected: {len(all_samples)}')
for cls in CLASSES:
    print(f'  {cls:<12} {counts[cls]}')

## Cell 6 — Auto-Annotate with MediaPipe Face Detector

MediaPipe detects faces and returns bounding boxes.  
If no face is detected, the full image is used as the bounding box.

In [ ]:
import cv2
import numpy as np

# OpenCV built-in face detector — no extra install needed
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

def get_yolo_bbox(image_path):
    """Returns (x_center, y_center, width, height) normalized to [0,1].
    Uses OpenCV Haar cascade face detector.
    Falls back to full image if no face detected."""
    img = cv2.imread(str(image_path))
    if img is None:
        return None
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=3, minSize=(20, 20)
    )

    if len(faces) > 0:
        fx, fy, fw, fh = faces[0]
        xc = (fx + fw / 2) / w
        yc = (fy + fh / 2) / h
        bw = fw / w
        bh = fh / h
        # clamp to [0,1]
        xc = max(0.0, min(1.0, xc))
        yc = max(0.0, min(1.0, yc))
        bw = max(0.01, min(1.0, bw))
        bh = max(0.01, min(1.0, bh))
        return (xc, yc, bw, bh)
    else:
        return (0.5, 0.5, 1.0, 1.0)  # full image fallback

# Test on one image
test_img, test_label = all_samples[0]
bbox = get_yolo_bbox(test_img)
print(f'Test image : {test_img.name}')
print(f'Label      : {test_label}')
print(f'YOLO bbox  : {bbox}')
print('Face detector ready (OpenCV Haar cascade)')

## Cell 7 — Build YOLO Dataset (Train / Val / Test Split)

In [ ]:
import random
from tqdm import tqdm

random.seed(42)

# Clean and create YOLO directory structure
if YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)
for split in ['train', 'val', 'test']:
    (YOLO_DIR / 'images' / split).mkdir(parents=True)
    (YOLO_DIR / 'labels' / split).mkdir(parents=True)

# Shuffle and split per class to keep class balance
class_samples = {cls: [] for cls in CLASSES}
for path, label in all_samples:
    class_samples[label].append(path)

split_counts = Counter()
failed       = 0

for cls, paths in class_samples.items():
    random.shuffle(paths)
    n      = len(paths)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)

    splits = (
        [('train', p) for p in paths[:n_train]] +
        [('val',   p) for p in paths[n_train:n_train + n_val]] +
        [('test',  p) for p in paths[n_train + n_val:]]
    )

    class_id = CLASS_TO_ID[cls]

    for split, src in tqdm(splits, desc=f'{cls}', leave=False):
        bbox = get_yolo_bbox(src)
        if bbox is None:
            failed += 1
            continue

        # Copy image
        dst_img = YOLO_DIR / 'images' / split / src.name
        # Avoid filename collisions across subjects
        if dst_img.exists():
            dst_img = dst_img.with_stem(dst_img.stem + f'_{cls[:3]}')
        shutil.copy2(src, dst_img)

        # Write YOLO label
        label_file = YOLO_DIR / 'labels' / split / (dst_img.stem + '.txt')
        xc, yc, bw, bh = bbox
        label_file.write_text(f'{class_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n')

        split_counts[f'{split}/{cls}'] += 1

print(f'\nFailed (unreadable): {failed}')
print('\nDataset split:')
for split in ['train', 'val', 'test']:
    total = sum(v for k, v in split_counts.items() if k.startswith(split))
    print(f'  {split:<6}  {total}  images')
    for cls in CLASSES:
        print(f'         {cls:<12} {split_counts[f"{split}/{cls}"]}')

## Cell 8 — Create data.yaml

In [ ]:
yaml_content = f"""path: {YOLO_DIR}
train: images/train
val:   images/val
test:  images/test

nc: {len(CLASSES)}
names: {CLASSES}
"""

yaml_path = YOLO_DIR / 'data.yaml'
yaml_path.write_text(yaml_content)

print('data.yaml created:')
print(yaml_content)

## Cell 9 — Verify Dataset (Optional Sanity Check)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

COLORS_VIS = {'No_Pain': '#2ecc71', 'Mild': '#f1c40f', 'Moderate': '#e67e22', 'Severe': '#e74c3c'}
ID_TO_CLASS = {i: c for i, c in enumerate(CLASSES)}

# Show 8 random training samples with bounding boxes
train_imgs = list((YOLO_DIR / 'images' / 'train').iterdir())[:8]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.patch.set_facecolor('#0f1117')

for ax, img_path in zip(axes.flat, train_imgs):
    lbl_path = YOLO_DIR / 'labels' / 'train' / (img_path.stem + '.txt')
    img = np.array(Image.open(img_path))
    h, w = img.shape[:2]

    ax.imshow(img)
    ax.set_facecolor('#0f1117')
    ax.axis('off')

    if lbl_path.exists():
        for line in lbl_path.read_text().strip().splitlines():
            cid, xc, yc, bw, bh = map(float, line.split())
            cls_name = ID_TO_CLASS[int(cid)]
            color    = COLORS_VIS[cls_name]
            x1 = (xc - bw / 2) * w
            y1 = (yc - bh / 2) * h
            rect = patches.Rectangle((x1, y1), bw * w, bh * h,
                                      linewidth=2, edgecolor=color, facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1 - 4, cls_name, color=color, fontsize=8, fontweight='bold')

plt.suptitle('Sample Training Images with YOLO Bounding Boxes', color='white', fontsize=14)
plt.tight_layout()
plt.savefig('/content/sample_annotations.png', dpi=120, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()
print('Saved: /content/sample_annotations.png')

## Cell 10 — Train YOLOv8

> Runtime must be set to **GPU** (Runtime → Change runtime type → T4 GPU)

In [ ]:
from ultralytics import YOLO
import torch

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Load YOLOv8 nano (fastest, good for student projects)
model = YOLO('yolov8n.pt')

results = model.train(
    data    = str(yaml_path),
    epochs  = 50,
    imgsz   = 224,
    batch   = 16,
    device  = 0 if torch.cuda.is_available() else 'cpu',
    project = str(RESULTS_DIR),
    name    = 'pain_detection',
    patience= 10,       # early stopping
    exist_ok= True,
    plots   = True,
    verbose = True,
)

print('\nTraining complete!')
BEST_WEIGHTS = RESULTS_DIR / 'pain_detection' / 'weights' / 'best.pt'
print(f'Best weights: {BEST_WEIGHTS}')

## Cell 11 — Evaluate on Test Set (mAP Results)

In [ ]:
import json

# Load best model
best_model = YOLO(str(BEST_WEIGHTS))

# Evaluate on test split
metrics = best_model.val(
    data   = str(yaml_path),
    split  = 'test',
    imgsz  = 224,
    device = 0 if torch.cuda.is_available() else 'cpu',
    plots  = True,
    save_json = True,
)

# Print results
print('=' * 50)
print('  Week 3 — mAP Results')
print('=' * 50)
print(f'  mAP@0.5       : {metrics.box.map50:.4f}  ({metrics.box.map50*100:.2f}%)')
print(f'  mAP@0.5:0.95  : {metrics.box.map:.4f}   ({metrics.box.map*100:.2f}%)')
print(f'  Precision      : {metrics.box.mp:.4f}')
print(f'  Recall         : {metrics.box.mr:.4f}')
print()
print('Per-class AP@0.5:')
for cls, ap in zip(CLASSES, metrics.box.ap50):
    print(f'  {cls:<12} {ap:.4f}  ({ap*100:.2f}%)')

# Save metrics JSON
map_results = {
    'mAP_50':      round(float(metrics.box.map50), 4),
    'mAP_50_95':   round(float(metrics.box.map),   4),
    'precision':   round(float(metrics.box.mp),    4),
    'recall':      round(float(metrics.box.mr),    4),
    'per_class_AP50': {
        cls: round(float(ap), 4)
        for cls, ap in zip(CLASSES, metrics.box.ap50)
    }
}
with open('/content/map_results.json', 'w') as f:
    json.dump(map_results, f, indent=2)
print('\nSaved: /content/map_results.json')

## Cell 12 — Visualize Predictions on Test Images

In [ ]:
import os

# Run predictions on test images
test_img_dir = YOLO_DIR / 'images' / 'test'
pred_results = best_model.predict(
    source  = str(test_img_dir),
    imgsz   = 224,
    conf    = 0.25,
    save    = True,
    project = '/content/predictions',
    name    = 'test_preds',
    exist_ok= True,
    device  = 0 if torch.cuda.is_available() else 'cpu',
)

print(f'Predictions saved to /content/predictions/test_preds/')

## Cell 13 — Plot Sample Predictions

In [ ]:
pred_dir = Path('/content/predictions/test_preds')
pred_imgs = sorted(pred_dir.glob('*.jpg'))[:12]

n = len(pred_imgs)
cols = 4
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
fig.patch.set_facecolor('#0f1117')

for ax, img_path in zip(axes.flat, pred_imgs):
    img = np.array(Image.open(img_path))
    ax.imshow(img)
    ax.set_title(img_path.name[:20], color='white', fontsize=8)
    ax.set_facecolor('#0f1117')
    ax.axis('off')

# Hide unused axes
for ax in axes.flat[n:]:
    ax.set_visible(False)

plt.suptitle('YOLOv8 Pain Detection — Test Set Predictions', color='white', fontsize=14)
plt.tight_layout()
plt.savefig('/content/prediction_grid.png', dpi=120, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()
print('Saved: /content/prediction_grid.png')

## Cell 14 — Training Curve (Loss & mAP over Epochs)

In [ ]:
import pandas as pd

csv_path = RESULTS_DIR / 'pain_detection' / 'results.csv'

if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.patch.set_facecolor('#0f1117')

    plot_cfg = [
        ('train/box_loss', 'val/box_loss',   'Box Loss',  '#7c6af7', '#e74c3c'),
        ('train/cls_loss', 'val/cls_loss',   'Class Loss','#2ecc71', '#f1c40f'),
        ('metrics/mAP50',  'metrics/mAP50-95','mAP',      '#3498db', '#e67e22'),
    ]

    for ax, (t_col, v_col, title, tc, vc) in zip(axes, plot_cfg):
        ax.set_facecolor('#1a1d27')
        epochs = range(1, len(df) + 1)
        if t_col in df.columns:
            ax.plot(epochs, df[t_col], color=tc, label='Train', linewidth=2)
        if v_col in df.columns:
            ax.plot(epochs, df[v_col], color=vc, label='Val', linewidth=2, linestyle='--')
        ax.set_title(title, color='white', fontsize=12)
        ax.set_xlabel('Epoch', color='#888')
        ax.tick_params(colors='#888')
        for spine in ax.spines.values():
            spine.set_edgecolor('#2a2d3e')
        ax.legend(facecolor='#1a1d27', labelcolor='white')
        ax.grid(True, color='#2a2d3e', linestyle='--', alpha=0.5)

    plt.suptitle('YOLOv8 Training Curves', color='white', fontsize=14)
    plt.tight_layout()
    plt.savefig('/content/yolo_training_curves.png', dpi=120, bbox_inches='tight',
                facecolor='#0f1117')
    plt.show()
    print('Saved: /content/yolo_training_curves.png')
else:
    print('results.csv not found — training may not have completed')

## Cell 15 — Summary Report

In [ ]:
print('=' * 55)
print('  WEEK 3 — Pain Expression Detection — Final Report')
print('=' * 55)
print()
print('Model        : YOLOv8n (fine-tuned)')
print(f'Dataset      : SZU-EmoDage ({len(all_samples)} images)')
print(f'Classes      : {CLASSES}')
print()
print('--- mAP Results (Test Set) ---')
print(f'  mAP@0.5       : {map_results["mAP_50"]*100:.2f}%')
print(f'  mAP@0.5:0.95  : {map_results["mAP_50_95"]*100:.2f}%')
print(f'  Precision      : {map_results["precision"]*100:.2f}%')
print(f'  Recall         : {map_results["recall"]*100:.2f}%')
print()
print('--- Per-class AP@0.5 ---')
for cls, ap in map_results['per_class_AP50'].items():
    bar = '#' * int(ap * 30)
    print(f'  {cls:<12} {ap*100:5.2f}%  {bar}')
print()
print('--- Week 2 vs Week 3 Comparison ---')
print('  Week 2 (MobileNetV2 Classification)  : 93.02% accuracy')
print(f'  Week 3 (YOLOv8 Detection) mAP@0.5   : {map_results["mAP_50"]*100:.2f}%')
print()
print('Note: mAP is stricter than accuracy — it also evaluates')
print('      bounding box quality (IoU), not just class prediction.')

## Cell 16 — Save All Results to Google Drive

In [ ]:
import shutil

DRIVE_OUT = Path('/content/drive/MyDrive/Week3_Results')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

# 1. Best model weights
shutil.copy2(BEST_WEIGHTS, DRIVE_OUT / 'best.pt')

# 2. mAP results JSON
shutil.copy2('/content/map_results.json', DRIVE_OUT / 'map_results.json')

# 3. Training curves
shutil.copy2('/content/yolo_training_curves.png', DRIVE_OUT / 'training_curves.png')

# 4. Sample annotation visualization
shutil.copy2('/content/sample_annotations.png', DRIVE_OUT / 'sample_annotations.png')

# 5. Prediction grid
shutil.copy2('/content/prediction_grid.png', DRIVE_OUT / 'prediction_grid.png')

# 6. YOLO results folder (confusion matrix, PR curve, etc.)
yolo_res = RESULTS_DIR / 'pain_detection'
if yolo_res.exists():
    shutil.copytree(yolo_res, DRIVE_OUT / 'yolo_results', dirs_exist_ok=True)

# 7. Sample prediction images (first 20)
pred_out = DRIVE_OUT / 'prediction_samples'
pred_out.mkdir(exist_ok=True)
for p in sorted(Path('/content/predictions/test_preds').glob('*.jpg'))[:20]:
    shutil.copy2(p, pred_out / p.name)

print('All results saved to Google Drive:')
print(f'  {DRIVE_OUT}')
for f in sorted(DRIVE_OUT.rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to(DRIVE_OUT)}')